# 04 — Distributed Parameter Sweep

This is the Spark showpiece: 45 backtest configurations executed in parallel via
Spark RDDs. Each task runs one backtest against a broadcast copy of the features
DataFrame. We then apply the Deflated Sharpe Ratio correction (Bailey & López de
Prado 2014) to account for the selection bias inherent in picking the best config
from a grid.


In [1]:
import sys, os, time
sys.path.insert(0, '..')
import pandas as pd
from src.spark_session import get_spark
from src.strategies import cross_sectional_momentum, mean_reversion
from src.backtest import run_backtest, compute_turnover
from src.metrics import sharpe, sortino, max_drawdown, calmar, cagr, deflated_sharpe

spark = get_spark('param_sweep', driver_mem='6g', shuffle_parts=16)
sc = spark.sparkContext

# Ship the src/ package to workers. Workers don't inherit driver sys.path.
import zipfile, shutil
src_zip = '/tmp/bigdata_trading_src.zip'
with zipfile.ZipFile(src_zip, 'w') as zf:
    for fname in ['__init__.py', 'strategies.py', 'backtest.py', 'metrics.py', 'features.py', 'spark_session.py']:
        zf.write(f'../src/{fname}', arcname=f'src/{fname}')
sc.addPyFile(src_zip)
print('Spark default parallelism:', sc.defaultParallelism, '| src shipped via addPyFile')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 05:20:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark default parallelism: 12 | src shipped via addPyFile


## Build config grid (27 momentum + 18 mean-reversion = 45)

In [2]:
mom_configs = []
cid = 0
for lb in ['mom_63d', 'mom_126d', 'mom_252d']:
    for tp in [0.10, 0.20, 0.30]:
        for rd in [5, 21, 63]:
            mom_configs.append({
                'config_id': f'mom_{cid:03d}', 'strategy': 'momentum',
                'lookback_col': lb, 'top_pct': tp, 'rebalance_days': rd,
            })
            cid += 1

mr_configs = []
cid = 0
for zc in ['bb_pct']:
    for ez in [1.0, 1.5, 2.0]:
        for xz in [0.0, 0.5]:
            for mh in [5, 10, 20]:
                mr_configs.append({
                    'config_id': f'mr_{cid:03d}', 'strategy': 'mean_reversion',
                    'z_col': zc, 'entry_z': ez, 'exit_z': xz, 'max_holding_days': mh,
                })
                cid += 1

configs = mom_configs + mr_configs
print(f'momentum configs: {len(mom_configs)} | mean-reversion configs: {len(mr_configs)}'
      f' | total: {len(configs)}')

momentum configs: 27 | mean-reversion configs: 18 | total: 45


## Load features + broadcast to workers

In [3]:
feats = pd.read_parquet('../data/parquet/features')
feats['date'] = pd.to_datetime(feats['date'])
print('features:', feats.shape, 'approx memory MB:',
      feats.memory_usage(deep=True).sum() / 1_048_576)

# Slim payload: only the columns any strategy needs
need_cols = ['date','ticker','simple_ret','mom_63d','mom_126d','mom_252d','bb_pct']
payload = feats[need_cols].copy()
print('payload:', payload.shape,
      'memory MB:', payload.memory_usage(deep=True).sum() / 1_048_576)

broadcast_payload = sc.broadcast(payload)
print('broadcast OK')

features: (1940547, 29) approx memory MB: 512.9447231292725
payload: (1940547, 7) memory MB: 200.18442821502686


broadcast OK


## Worker function — runs one config, returns metrics row

In [4]:
def run_one_config(cfg):
    import pandas as pd
    from src.strategies import cross_sectional_momentum, mean_reversion
    from src.backtest import run_backtest, compute_turnover
    from src.metrics import sharpe, sortino, max_drawdown, calmar, cagr

    feats_local = broadcast_payload.value
    returns = feats_local[['date','ticker','simple_ret']].dropna()

    try:
        if cfg['strategy'] == 'momentum':
            w = cross_sectional_momentum(
                feats_local,
                lookback_col=cfg['lookback_col'],
                top_pct=cfg['top_pct'],
                rebalance_days=cfg['rebalance_days'],
            )
        else:
            w = mean_reversion(
                feats_local,
                z_col=cfg['z_col'],
                entry_z=cfg['entry_z'],
                exit_z=cfg['exit_z'],
                max_holding_days=cfg['max_holding_days'],
            )
        bt = run_backtest(w, returns)
        n_trades = int((w.groupby('date')['weight'].apply(lambda x: (x!=0).sum())).sum())
        return {
            **cfg,
            'sharpe': sharpe(bt['net_ret']),
            'sortino': sortino(bt['net_ret']),
            'max_dd': max_drawdown(bt['equity']),
            'calmar': calmar(bt['net_ret'], bt['equity']),
            'cagr': cagr(bt['equity']),
            'turnover': compute_turnover(w),
            'n_position_rows': n_trades,
            'error': '',
        }
    except Exception as exc:
        return {**cfg, 'sharpe': float('nan'), 'sortino': float('nan'),
                'max_dd': float('nan'), 'calmar': float('nan'),
                'cagr': float('nan'), 'turnover': float('nan'),
                'n_position_rows': 0, 'error': repr(exc)}

## Run the distributed sweep (all 45 configs in parallel)

In [5]:
n_configs = len(configs)
rdd = sc.parallelize(configs, numSlices=n_configs)

t0 = time.time()
results = rdd.map(run_one_config).collect()
parallel_elapsed = time.time() - t0

results_df = pd.DataFrame(results)
ok = results_df['error'] == ''
print(f'Parallel sweep: {n_configs} configs in {parallel_elapsed:.1f}s '
      f'= {n_configs/parallel_elapsed:.2f} configs/sec')
print(f'OK: {ok.sum()} | errored: {(~ok).sum()}')
if (~ok).any():
    print(results_df[~ok][['config_id','error']])

Parallel sweep: 45 configs in 19.4s = 2.32 configs/sec
OK: 45 | errored: 0


## Sequential baseline for comparison
Run 3 configs in a driver loop (no Spark parallelism) and extrapolate.

In [6]:
seq_sample = configs[:3]
t0 = time.time()
seq_results = [run_one_config(c) for c in seq_sample]
seq_elapsed_3 = time.time() - t0
seq_per_config = seq_elapsed_3 / len(seq_sample)
seq_elapsed_45_est = seq_per_config * n_configs
speedup = seq_elapsed_45_est / parallel_elapsed
print(f'Sequential (3 configs)     : {seq_elapsed_3:.2f}s -> {seq_per_config:.2f}s/config')
print(f'Sequential (45 configs est): {seq_elapsed_45_est:.2f}s')
print(f'Parallel (45 configs meas) : {parallel_elapsed:.2f}s')
print(f'Speedup                    : {speedup:.2f}x')

Sequential (3 configs)     : 4.33s -> 1.44s/config
Sequential (45 configs est): 65.01s
Parallel (45 configs meas) : 19.36s
Speedup                    : 3.36x


## Rank configs and apply DSR correction

In [7]:
top = (results_df[results_df['error']=='']
       .sort_values('sharpe', ascending=False)
       .head(10).copy())
print('Top 10 by raw Sharpe:')
print(top[['config_id','strategy','sharpe','sortino','max_dd','cagr','turnover']]
      .round(4).to_string(index=False))

Top 10 by raw Sharpe:
config_id       strategy  sharpe  sortino  max_dd   cagr  turnover
  mom_011       momentum  0.2704   0.2521 -0.4812 0.0348    0.0390
  mom_020       momentum  0.2162   0.1964 -0.5032 0.0233    0.0284
  mom_019       momentum  0.2105   0.1895 -0.6599 0.0219    0.0502
  mom_014       momentum  0.1797   0.1675 -0.3958 0.0160    0.0338
   mr_014 mean_reversion  0.1693   0.1905 -0.5036 0.0146    0.6500
  mom_023       momentum  0.1635   0.1484 -0.4124 0.0133    0.0244
  mom_026       momentum  0.1583   0.1437 -0.3167 0.0123    0.0209
  mom_022       momentum  0.1576   0.1417 -0.5361 0.0122    0.0425
   mr_008 mean_reversion  0.1434   0.1514 -0.5077 0.0102    0.6885
  mom_017       momentum  0.1373   0.1283 -0.3519 0.0094    0.0295


In [8]:
# To apply DSR we need a returns series for each config. We re-run the top 5.
returns_all = feats[['date','ticker','simple_ret']].dropna()
def rerun_returns(cfg):
    from src.strategies import cross_sectional_momentum, mean_reversion
    # iterrows() upcasts ints to float64 when any cell is NaN; coerce back.
    if cfg['strategy']=='momentum':
        w = cross_sectional_momentum(feats, lookback_col=cfg['lookback_col'],
                                     top_pct=float(cfg['top_pct']),
                                     rebalance_days=int(cfg['rebalance_days']))
    else:
        w = mean_reversion(feats, z_col=cfg['z_col'],
                            entry_z=float(cfg['entry_z']), exit_z=float(cfg['exit_z']),
                            max_holding_days=int(cfg['max_holding_days']))
    bt = run_backtest(w, returns_all)
    return bt['net_ret']

dsr_rows = []
for _, row in top.head(5).iterrows():
    cfg = row.to_dict()
    rets = rerun_returns(cfg)
    p = deflated_sharpe(row['sharpe'], n_trials=n_configs, returns=rets.values)
    dsr_rows.append({'config_id': row['config_id'],
                     'raw_sharpe': row['sharpe'], 'dsr_prob': p})

dsr_df = pd.DataFrame(dsr_rows)
print('\nDSR-corrected top 5 (probability that observed Sharpe exceeds expected-max-under-null):')
print(dsr_df.round(4).to_string(index=False))


DSR-corrected top 5 (probability that observed Sharpe exceeds expected-max-under-null):
config_id  raw_sharpe  dsr_prob
  mom_011      0.2704       0.0
  mom_020      0.2162       0.0
  mom_019      0.2105       0.0
  mom_014      0.1797       0.0
   mr_014      0.1693       0.0


## Save results

In [9]:
import os
os.makedirs('../reports', exist_ok=True)
results_df.to_csv('../reports/parameter_sweep_results.csv', index=False)
results_df.to_parquet('../reports/parameter_sweep_results.parquet', index=False)
dsr_df.to_csv('../reports/dsr_top5.csv', index=False)
print('wrote reports/parameter_sweep_results.csv (+ parquet) + reports/dsr_top5.csv')

wrote reports/parameter_sweep_results.csv (+ parquet) + reports/dsr_top5.csv


In [10]:
spark.stop()